In [2]:
# ==============================================================================
# IMPORTS
# ==============================================================================
# PySpark SQL
from pyspark.sql import SparkSession
import pyspark.sql.functions as sql_f

# PySpark ML - Features, Models & Evaluation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.linalg import Vectors

# Configuración local
from config import (
    TRAIN_TEST_SPLIT,
    RANDOM_SEED,
    INITIAL_LABELED_FRACTION,
    QUERY_BATCH_FRACTION,
    UNCERTAINTY_CANDIDATES_MULTIPLIER,
)

In [4]:
print("\n" + "=" * 60)
print(" STARTING SPARK ACTIVE LEARNING")
print("=" * 60)
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ActiveLearning")
    .getOrCreate()
)

# Obtenemos el SparkContext asociado a la SparkSession
sc = spark.sparkContext
print(f"  -> Spark session initialized successfully on '{sc.master}'")


 STARTING SPARK ACTIVE LEARNING
  -> Spark session initialized successfully on 'local[*]'


In [ ]:
# ==============================================================================
# DATA LOADING & PREPROCESSING
# ==============================================================================
# --- Step 2: Load Dataset ---
print("\n[2/N] Loading raw dataset...")
dataset_path = "data/susy-10.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv(dataset_path)
)
print(f"  -> Dataset loaded successfully from '{dataset_path}'")

# --- Step 3: Schema Casting ---
print("\n[3/N] Casting column data types...")

# Identify feature columns dynamically
feature_cols = [c for c in df.columns if c.startswith("feature_")]

# Casting variables
cast_expressions = [
    sql_f.col("id_sample").cast("int"),
    sql_f.col("label").cast("int")
] + [sql_f.col(c).cast("float") for c in feature_cols]
print(f"  -> Cast 'id_sample' and 'label' to INT")
print(f"  -> Cast {len(feature_cols)} feature columns to FLOAT")

# --- Step 4: Vector Assembly ---
print("\n[4/N] Assembling feature vector...")

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Assemble feature vector and drop original raw columns
df = assembler.transform(df).drop(*feature_cols)

print("  -> Assembled feature vector into 'features' column")
print("  -> Raw feature columns dropped successfully")


[2/N] Loading raw dataset...
  -> Dataset loaded successfully from 'data/susy-10.csv'

[3/N] Casting column data types...
  -> Cast 'id_sample' and 'label' to INT
  -> Cast 18 feature columns to FLOAT

[4/N] Assembling feature vector...
  -> Assembled feature vector into 'features' column
  -> Raw feature columns dropped successfully


In [11]:
# ==============================================================================
# TRAIN / TEST SPLIT & INITIAL LABELED POOL CREATION
# ==============================================================================
# --- Step 5: Train / Test Split ---
print("\n[5/N] Splitting dataset into training and test sets...")

train_df, test_df = df.randomSplit(
    TRAIN_TEST_SPLIT,
    seed=RANDOM_SEED
)

# Randomly assign initial state: 'L' (Labeled) or 'U' (Unlabeled)
train_df = train_df.withColumn(
    "state",
    sql_f.when(
        sql_f.rand(RANDOM_SEED) < INITIAL_LABELED_FRACTION,
        "L"
    ).otherwise("U")
)
print(f"  -> Train/Test split ratio applied: {TRAIN_TEST_SPLIT}")
print(f"  -> Initial labeled fraction configured: {INITIAL_LABELED_FRACTION * 100:.1f}%")

# --- Step 6: Caching & Materializing DataFrames ---
print("\n[6/N] Caching DataFrames and calculating initial stats...")

# Persist both sets in memory to avoid recomputing upstream transformations
train_df.cache()
test_df.cache()

# Materialize train_df cache and compute labeled/unlabeled counts in a single action
stats = train_df.select(
    sql_f.count(sql_f.when(sql_f.col("state") == "L", True)).alias("labeled"),
    sql_f.count(sql_f.when(sql_f.col("state") == "U", True)).alias("unlabeled")
).first()

labeled_size = stats["labeled"]
unlabeled_size = stats["unlabeled"]
train_size = labeled_size + unlabeled_size

# Materialize test_df cache
test_size = test_df.count()

# Display split breakdown
print(f"  -> Total Train dataset size : {train_size} rows")
print(f"     ├── Initial Labeled (L)  : {labeled_size} rows ({labeled_size / train_size * 100:.2f}%)")
print(f"     └── Unlabeled pool (U)   : {unlabeled_size} rows ({unlabeled_size / train_size * 100:.2f}%)")
print(f"  -> Total Test dataset size  : {test_size:,} rows")
print("  -> DataFrames successfully cached in memory.")


[5/N] Splitting dataset into training and test sets...
  -> Train/Test split ratio applied: [0.7, 0.3]
  -> Initial labeled fraction configured: 5.0%

[6/N] Caching DataFrames and calculating initial stats...
  -> Total Train dataset size : 70262 rows
     ├── Initial Labeled (L)  : 3490 rows (4.97%)
     └── Unlabeled pool (U)   : 66772 rows (95.03%)
  -> Total Test dataset size  : 29,738 rows
  -> DataFrames successfully cached in memory.


In [14]:
def train_and_evaluate_model(labeled_train_df, test_df, label_col="label", features_col="features"):
    """
    Trains a Logistic Regression model on the training set and evaluates accuracy on the test set.

    Parameters:
        labeled_train_df (DataFrame): PySpark DataFrame with labeled training instances.
        test_df (DataFrame): PySpark DataFrame with test instances.
        label_col (str): Name of the target label column. Default is 'label'.
        features_col (str): Name of the feature vector column. Default is 'features'.

    Returns:
        lr_model: Trained LogisticRegressionModel instance.
        accuracy (float): Test set accuracy score.
    """
    print("  -> Training Logistic Regression model...")
    lr = LogisticRegression(
        labelCol=label_col,
        featuresCol=features_col
    )
    lr_model = lr.fit(labeled_train_df)

    print("  -> Evaluating model predictions on test set...")
    # Generate predictions on test set
    test_predictions = lr_model.transform(test_df)

    # Evaluate Accuracy
    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="accuracy"
    )

    accuracy = accuracy_evaluator.evaluate(test_predictions)
    print(f"  -> Test Accuracy: {accuracy:.4f}")

    return lr_model, accuracy


# Filter labeled dataset for training (state == 'L')
labeled_train_df = train_df.filter(sql_f.col("state") == "L")

# Train and evaluate model
lr_model, test_accuracy = train_and_evaluate_model(labeled_train_df, test_df)

  -> Training Logistic Regression model...
  -> Evaluating model predictions on test set...
  -> Test Accuracy: 0.7891


In [16]:
def get_uncertainty_candidates(unlabeled_pred_df, p, epsilon= 0.001):
    """
    Selects top candidate instances from the unlabeled pool with the highest uncertainty
    using distributed quantile estimation (Greenwald-Khanna algorithm).

    Parameters:
        unlabeled_pred_df (DataFrame): PySpark DataFrame containing predictions and 
                                       an 'uncertainty' metric column.
        p (float): Fraction of top candidate instances to select.
        epsilon (float): Relative error parameter for approxQuantile. Default is 0.001.

    Returns:
        DataFrame: PySpark DataFrame containing ('id_sample', 'features', 'uncertainty') 
                   for the filtered uncertainty candidates.
    """
    print(f"  -> Computing distributed uncertainty threshold (quantile target: {1.0 - p:.4f})...")
    unlabeled_pred_df = unlabeled_pred_df.withColumn(
        "uncertainty",
        1 - 2 * sql_f.abs(
            vector_to_array("probability")[1] - 0.5
        )
    )
    # Distributed quantile calculation using Greenwald-Khanna algorithm
    threshold_val = unlabeled_pred_df.stat.approxQuantile(
        "uncertainty", [1.0 - p], epsilon
    )[0]

    print(f"  -> Calculated uncertainty threshold: {threshold_val:.6f}")

    # Filter candidated instances
    uncertainty_candidates_df = (
        unlabeled_pred_df
        .select("id_sample", "features", "uncertainty")
        .filter(sql_f.col("uncertainty") >= threshold_val)
    )

    return uncertainty_candidates_df

unlabeled_df = train_df.filter(sql_f.col("state") == "U")
unlabeled_pred_df = lr_model.transform(unlabeled_df)

# Parametros de configuracion del metodo:
# No podemos pedir mas muestras al oráculo de las que quedan en U
# Hacer la operacion train_size * Query_BATCH_FRACTION al inicio
# almacenar en una variable llamada B y mostrar en configuracion
query_batch = min(
    int(train_size * QUERY_BATCH_FRACTION),
    unlabeled_size
)

# Proporción de muestras a solicitar
p = min(1.0, (UNCERTAINTY_CANDIDATES_MULTIPLIER * query_batch) / unlabeled_size)

# Select candidate pool based on uncertainty threshold
uncertainty_candidates_df = get_uncertainty_candidates(
    unlabeled_pred_df=unlabeled_pred_df,
    p=p,
    epsilon=0.001
)

  -> Computing distributed uncertainty threshold (quantile target: 0.8949)...
  -> Calculated uncertainty threshold: 0.841607


In [17]:
print(
    "Uncertainty candidates DataFrame partitions:",
    uncertainty_candidates_df.rdd.getNumPartitions()
)

Uncertainty candidates DataFrame partitions: 9


In [18]:
uncertainty_candidates_df.count()

7087

In [ ]:
def diversity_k_means_selection(
    uncertainty_candidates_df,
    train_df,
    query_batch,
    spark,
    seed
):
    """
    Applies KMeans clustering on uncertainty candidates to select diverse instances 
    closest to each cluster centroid, updating their state in train_df to 'L' (Labeled).

    Parameters:
        uncertainty_candidates_df (DataFrame): PySpark DataFrame containing candidate rows.
        train_df (DataFrame): Main training PySpark DataFrame with 'id_sample' and 'state' columns.
        query_batch (int): Number of clusters (k) to form and instances to select.
        spark (SparkSession): Active SparkSession instance.
        seed (int): Random seed for KMeans initialization. Default is RANDOM_SEED.

    Returns:
        DataFrame: Updated train_df with selected target instances marked as 'L'.
    """
    print(f"  -> Running KMeans clustering (k={query_batch})...")

    # Initialize and fit KMeans model
    kmeans = KMeans(
        k=query_batch,
        seed=seed,
        featuresCol="features",
        predictionCol="cluster_id",
        maxIter=20,
    )

    kmeans_model = kmeans.fit(uncertainty_candidates_df)
    clustered_candidates_df = kmeans_model.transform(uncertainty_candidates_df)

    # Broadcast cluster centroids to worker nodes
    sc = spark.sparkContext
    centers_broadcast = sc.broadcast(kmeans_model.clusterCenters())

    print("  -> Finding closest sample instance to each cluster centroid...")

    # 1. Map DataFrame to (cluster_id, (id_sample, squared_distance_to_centroid))
    rdd_mapped = clustered_candidates_df.rdd.map(
        lambda row: (
            row.cluster_id,
            (
                row.id_sample,
                float(Vectors.squared_distance(row.features, centers_broadcast.value[row.cluster_id]))
            )
        )
    )

    # 2. Reduce by key to pick the minimum distance candidate per cluster
    closest_per_cluster_rdd = rdd_mapped.reduceByKey(
        lambda candidate1, candidate2: candidate1 if candidate1[1] < candidate2[1] else candidate2
    )

    # Extract target IDs into a DataFrame
    target_ids_df = closest_per_cluster_rdd.map(lambda x: (x[1][0],)).toDF(["target_id"])

    # 3. Update 'state' column in train_df via Left Join
    updated_train_df = (
        train_df.join(
            target_ids_df,
            train_df["id_sample"] == target_ids_df["target_id"],
            how="left"
        )
        .withColumn(
            "state",
            sql_f.when(sql_f.col("target_id").isNotNull(), sql_f.lit("L")).otherwise(sql_f.col("state"))
        )
        .drop("target_id")
    )

    # Release broadcast variable from memory
    centers_broadcast.destroy()
    updated_train_df = updated_train_df.localCheckpoint()
    print(f"  -> Successfully updated {query_batch} instances from 'U' to 'L' state.")

    return updated_train_df

In [22]:
# 1. Guardar una referencia al DataFrame antiguo para poder liberar su caché
old_train_df = train_df

train_df = diversity_k_means_selection(
    uncertainty_candidates_df=uncertainty_candidates_df,
    train_df=train_df,
    query_batch=query_batch,
    spark=spark,
    seed= RANDOM_SEED
)
# 3. Cachear el nuevo DataFrame
train_df.cache()
# 5. Liberar la caché del DataFrame anterior de la RAM
old_train_df.unpersist()

  -> Running KMeans clustering (k=702)...
  -> Finding closest sample instance to each cluster centroid...
  -> Successfully updated 702 instances from 'U' to 'L' state.


DataFrame[id_sample: int, label: double, features: vector, state: string]

In [23]:
# Filter labeled dataset for training (state == 'L')
labeled_train_df = train_df.filter(sql_f.col("state") == "L")

# Train and evaluate model
lr_model, test_accuracy = train_and_evaluate_model(labeled_train_df, test_df)

  -> Training Logistic Regression model...
  -> Evaluating model predictions on test set...
  -> Test Accuracy: 0.7908
